# Лекция 07. Хеш-таблицы

Хеш-таблица решает очень бытовую задачу: быстро найти запись по известному ключу. Никакой магии внутри нет — массив корзин, функция выбора корзины и аккуратная обработка совпадений.

## Цели

После лекции вы сможете:

- объяснять путь от ключа до корзины;
- различать равенство ключей и коллизию;
- объяснять среднюю и худшую сложность операций;
- связывать коэффициент заполнения с resize;
- применять `dict` и `set` для подсчёта, дедупликации, индексирования и кэширования;
- реализовывать учебную таблицу методом цепочек;
- проверять коллизии, замену, удаление и рехеширование.

## Перед началом

Нужны списки, циклы, функции, `dict`, `set`, равенство и хешируемость из занятия 5, а также оценки сложности из занятия 3. На обсуждение устройства таблицы и кода заложено около 55 минут, на прикладные шаблоны — 20 минут, на контринтуитивные примеры, самопроверку и вопросы — 15 минут.

## Сначала реальная задача

Есть небольшой справочник курсов валют. Если он хранится списком, для поиска `USD` приходится просматривать записи по очереди. Один поиск на трёх строках незаметен, но миллион поисков по большому справочнику превращается в настоящую работу.

In [ ]:
rates = [("EUR", 101.2), ("USD", 92.4), ("CNY", 12.7)]

def find_rate(records, code):
    for stored_code, rate in records:
        if stored_code == code:
            return rate
    raise KeyError(code)

assert find_rate(rates, "USD") == 92.4

## Индекс меняет цену повторных поисков

Можно один раз построить словарь `код -> курс`, потратив `O(N)`, а затем обычно получать значение за `O(1)`. Индекс занимает дополнительную память, зато повторный поиск больше не зависит линейно от размера исходного списка.

In [ ]:
rate_by_code = {code: rate for code, rate in rates}
assert rate_by_code["USD"] == 92.4
assert "GBP" not in rate_by_code

## Модель хеш-таблицы

Упрощённый путь поиска состоит из четырёх шагов:

1. получить хеш ключа: `hash(key)`;
2. выбрать номер корзины, например `hash(key) % capacity`;
3. просмотреть только выбранную корзину;
4. подтвердить совпадение ключей через `==`.

Хеш не является адресом объекта и не обязан быть уникальным. Он лишь быстро сужает множество кандидатов.

In [ ]:
def bucket_index(key, capacity):
    return hash(key) % capacity

capacity = 8
for code in ["EUR", "USD", "CNY"]:
    print(code, hash(code), bucket_index(code, capacity))

## Контракт хеша и равенства

Если `a == b`, обязательно должно выполняться `hash(a) == hash(b)`. Обратное неверно: одинаковый хеш ещё не доказывает равенство. Поэтому таблица сначала выбирает корзину по хешу, а затем сравнивает реальные ключи.

Ключ должен оставаться хешируемым и не менять смысл равенства, пока лежит в таблице. Именно поэтому список нельзя сделать ключом обычного словаря.

## Коллизии неизбежны

Ключей потенциально бесконечно много, а корзин конечное число. Разные ключи когда-нибудь попадут в одну корзину — это коллизия. Корректная таблица не пытается запретить коллизии, а хранит все столкнувшиеся пары и различает их через `==`.

В учебной реализации используем **метод цепочек**: каждая корзина является маленьким списком пар `[key, value]`. Реальные реализации могут применять другие стратегии; модель цепочек нужна, чтобы увидеть основной механизм.

## Создаём пустую таблицу

Таблица — список независимых корзин. Запись `[[]] * capacity` здесь была бы ошибкой: все позиции ссылались бы на один список. Comprehension создаёт отдельную корзину для каждой позиции.

In [ ]:
def make_table(capacity=8):
    if capacity <= 0:
        raise ValueError("capacity must be positive")
    return [[] for _ in range(capacity)]

table = make_table(4)
table[0].append(["test", 1])
assert table[1] == []

## Вставка и поиск

При вставке сначала ищем равный ключ в нужной корзине. Если нашли — заменяем значение. Если нет — добавляем новую пару. Поиск повторяет тот же маршрут и поднимает `KeyError`, когда корзина просмотрена до конца.

In [ ]:
def table_put(table, key, value):
    bucket = table[hash(key) % len(table)]
    for pair in bucket:
        if pair[0] == key:
            pair[1] = value
            return False
    bucket.append([key, value])
    return True

def table_get(table, key):
    bucket = table[hash(key) % len(table)]
    for stored_key, value in bucket:
        if stored_key == key:
            return value
    raise KeyError(key)

table = make_table(2)
assert table_put(table, 1, "one") is True
assert table_put(table, 3, "three") is True
assert table_get(table, 1) == "one"
assert table_get(table, 3) == "three"

### Что произошло с ключами `1` и `3`

Для небольшой демонстрации `hash(1) % 2` и `hash(3) % 2` равны. Обе пары лежат в одной цепочке, но поиск не путает их, потому что после выбора корзины сравнивает ключи. Коллизия ухудшила локальный поиск, но не нарушила корректность.

In [ ]:
print(table)
assert hash(1) % 2 == hash(3) % 2
assert 1 != 3

## Удаление

Удаление тоже работает только внутри одной корзины. Для метода цепочек достаточно найти индекс пары и удалить её из списка. Другие способы разрешения коллизий могут требовать специальных отметок вместо простого освобождения позиции.

In [ ]:
def table_delete(table, key):
    bucket = table[hash(key) % len(table)]
    for index, pair in enumerate(bucket):
        if pair[0] == key:
            del bucket[index]
            return True
    return False

assert table_delete(table, 1) is True
assert table_delete(table, 1) is False
assert table_get(table, 3) == "three"

## Коэффициент заполнения

Обозначим число пар через `size`, число корзин через `capacity`. Коэффициент заполнения

`load_factor = size / capacity`.

Если пар становится намного больше корзин, средние цепочки растут и поиск всё чаще просматривает несколько элементов. Таблица увеличивает массив корзин, а затем заново размещает пары. Точный порог — деталь конкретной реализации; важен сам принцип контроля заполнения.

In [ ]:
def table_size(table):
    return sum(len(bucket) for bucket in table)

def load_factor(table):
    return table_size(table) / len(table)

assert load_factor([[['a', 1]], [], [['b', 2]], []]) == 0.5

## Resize означает рехеширование

Нельзя просто дописать пустые корзины в конец. Номер вычисляется по модулю `capacity`, поэтому после изменения размера он может стать другим. Каждую старую пару надо вставить в новую таблицу обычным способом.

In [ ]:
def table_resize(table, new_capacity):
    new_table = make_table(new_capacity)
    for bucket in table:
        for key, value in bucket:
            table_put(new_table, key, value)
    return new_table

table = make_table(2)
for key, value in [(1, "one"), (3, "three"), (5, "five")]:
    table_put(table, key, value)
resized = table_resize(table, 8)
assert [table_get(resized, key) for key in (1, 3, 5)] == ["one", "three", "five"]

## Откуда берётся сложность

| Операция | В среднем при хорошем распределении | Худший случай |
|---|---:|---:|
| поиск по ключу | `O(1)` | `O(N)` |
| вставка или замена | `O(1)` | `O(N)` |
| удаление | `O(1)` | `O(N)` |
| обход всех пар | `O(N)` | `O(N)` |
| resize | `O(N)` за одно расширение | `O(N)` |

`O(1)` не означает «одна машинная инструкция». Оно означает, что средняя работа с одной корзиной не растёт вместе с общим числом записей. Редкий resize дорог, но его стоимость распределяется по множеству вставок — это амортизированный анализ.

## Шаблон 1. Частотный анализ

Нужно посчитать расходы по категориям. Словарь хранит накопленную сумму для каждой категории, а список просматривается один раз.

In [ ]:
transactions = [
    {"category": "еда", "amount": 900},
    {"category": "такси", "amount": 450},
    {"category": "еда", "amount": 300},
]

total_by_category = {}
for transaction in transactions:
    category = transaction["category"]
    total_by_category[category] = total_by_category.get(category, 0) + transaction["amount"]

assert total_by_category == {"еда": 1200, "такси": 450}

## Шаблон 2. Дедупликация с сохранением порядка

Множество быстро отвечает «уже видели?», а список сохраняет требуемый порядок. Просто вызвать `set(client_ids)` недостаточно, если порядок первого появления является частью результата.

In [ ]:
client_ids = ["c-2", "c-1", "c-2", "c-3", "c-1"]
seen = set()
unique = []
for client_id in client_ids:
    if client_id not in seen:
        seen.add(client_id)
        unique.append(client_id)

assert unique == ["c-2", "c-1", "c-3"]

## Шаблон 3. Индекс вместо вложенного цикла

Нужно присоединить баланс к профилю клиента. Вложенный поиск даёт `O(NM)`. Если сначала построить индекс балансов, вся работа занимает ожидаемое `O(N + M)`.

In [ ]:
profiles = [
    {"client_id": "c-1", "name": "Анна"},
    {"client_id": "c-2", "name": "Борис"},
]
balances = [
    {"client_id": "c-2", "balance": 300},
    {"client_id": "c-1", "balance": 120},
]

balance_by_client = {row["client_id"]: row["balance"] for row in balances}
report = [
    profile | {"balance": balance_by_client[profile["client_id"]]}
    for profile in profiles
]
assert report[0]["balance"] == 120

## Шаблон 4. Простой кэш

Если одна и та же чистая функция вызывается с повторяющимися аргументами, словарь может хранить уже вычисленный результат. Ключ должен полностью описывать вход, иначе кэш вернёт не тот ответ. Также нужно заранее решить, когда устаревшие значения удаляются.

In [ ]:
cache = {}

def net_amount(amount, tax_rate):
    key = (amount, tax_rate)
    if key not in cache:
        cache[key] = amount * (1 - tax_rate)
    return cache[key]

assert net_amount(1000, 0.13) == 870
assert (1000, 0.13) in cache

## `list`, `set` или `dict`

| Нужная операция | Естественный контейнер |
|---|---|
| сохранить последовательность и повторения | `list` |
| быстро проверить наличие значения | `set` |
| быстро найти значение по ключу | `dict` |
| сохранить порядок и быстро проверять повторы | `list` + `set` |
| присоединить записи по идентификатору | индекс в `dict` |

Контейнер выбирают не по привычке, а по основной операции и требуемому контракту результата.

## Что таблица не обещает

- Худший случай не становится `O(1)`: много ключей может попасть в одну корзину.
- Хеш строки не предназначен для хранения как постоянный идентификатор или криптографическая подпись.
- Множество не обещает сортировку или порядок вставки.
- Изменяемый объект с меняющимся хешем нельзя безопасно использовать как ключ.
- Учебный метод цепочек объясняет принцип, но не описывает внутренние детали конкретной версии CPython.

## Неожиданно, но по правилам

Сначала предскажите результат, затем свяжите его с контрактом хеша, порядком контейнера или рехешированием.

### 1. Коллизия не означает равенство

Два разных ключа могут оказаться в одной корзине. Это штатное состояние: номер корзины только выбирает короткий список кандидатов, а окончательное решение принимает `==`.

In [ ]:
assert hash(1) % 2 == hash(3) % 2
assert 1 != 3
table = make_table(2)
table_put(table, 1, "first")
table_put(table, 3, "second")
assert table_get(table, 1) != table_get(table, 3)

### 2. Порядок словаря — не порядок хешей

Современный `dict` перебирается в порядке вставки. Замена значения не переносит ключ в конец, а удаление и повторная вставка переносит.

In [ ]:
status = {"new": 3, "done": 8, "error": 1}
status["done"] = 9
assert list(status) == ["new", "done", "error"]
del status["done"]
status["done"] = 10
assert list(status) == ["new", "error", "done"]

### 3. Множество не является «списком без повторов»

`set` хранит уникальность, но не обещает ни порядок первого появления, ни сортировку. Если порядок важен, одного множества недостаточно.

In [ ]:
source = ["c-2", "c-1", "c-2", "c-3"]
unique_as_set = set(source)
assert unique_as_set == {"c-1", "c-2", "c-3"}
# На конкретный порядок list(unique_as_set) полагаться нельзя.

### 4. После resize ключ может сменить корзину

Сам ключ и его хеш не изменились. Изменился делитель в операции `% capacity`, поэтому прежний номер корзины больше не годится.

In [ ]:
key = 13
assert hash(key) % 4 != hash(key) % 8
small = make_table(4)
table_put(small, key, "payment")
large = table_resize(small, 8)
assert table_get(large, key) == "payment"

### 5. `hash("USD")` не обязан переживать перезапуск Python

Внутри одного процесса хеш остаётся согласованным, иначе поиск не работал бы. Но хеши `str` и `bytes` рандомизируются между процессами. Поэтому результат `hash()` нельзя записывать в файл как постоянный идентификатор.

In [ ]:
first = hash("USD")
second = hash("USD")
assert first == second
print("Хеш в текущем процессе:", first)

## Самопроверка

1. Зачем после совпадения корзины ещё сравнивать ключи через `==`?
2. Почему коллизия не является ошибкой?
3. Откуда берутся средняя `O(1)` и худшая `O(N)`?
4. Почему при resize нельзя просто перенести старые корзины как есть?
5. Когда для дедупликации нужны одновременно список и множество?
6. Как индекс превращает присоединение двух списков из `O(NM)` в ожидаемое `O(N + M)`?
7. Какие случаи обязательно проверить в собственной таблице?

## Источники

- [Встроенные типы Python: `dict`](https://docs.python.org/3/library/stdtypes.html#mapping-types-dict) — операции отображений, требования к ключам и порядок вставки.
- [Модель данных Python: `object.__hash__`](https://docs.python.org/3/reference/datamodel.html#object.__hash__) — контракт равенства и хеша, а также рандомизация хешей строк и байтов.
- [Design and History FAQ: dictionaries](https://docs.python.org/3/faq/design.html#how-are-dictionaries-implemented-in-cpython) — обзор связи хеша, поиска и коллизий в CPython.

## Итоги

- Хеш быстро выбирает корзину, но равенство ключей подтверждает `==`.
- Коллизии неизбежны; корректная таблица разрешает их без потери данных.
- Средняя работа с ключом близка к `O(1)` при контролируемом заполнении, худшая остаётся `O(N)`.
- Resize требует рехеширования, потому что номер корзины зависит от ёмкости.
- `dict` и `set` превращают подсчёт, дедупликацию, индексирование и кэширование в простые прикладные циклы.
- Учебная таблица состоит из знакомых списков и функций: никакого rocket science внутри нет.